# XGBoost: scale_pos_weight 베이스라인 vs SMOTE (DEATH) 비교

- **train/val/test**는 이미 분리된 CSV(`train.csv`, `val.csv`, `test.csv`)로 가정
- `feature_cols.json`에 feature column 리스트
- `scale_pos_weights.json`에 타깃별 scale_pos_weight
- Baseline pkl(이미 scale_pos_weight로 학습한 모델) 로드 후 동일 지표로 비교

> **중요:** SMOTE는 **train에만** 적용, SMOTE 모델 학습 시 **scale_pos_weight는 사용하지 않음(=1)**

In [ ]:

# ============================================
# Step 0: 라이브러리 & 설정
# ============================================
import os, json, pickle
from pathlib import Path

import numpy as np
import pandas as pd

from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import average_precision_score

# ✅ (필수) 경로만 본인 환경에 맞게 수정하세요.
DATA_DIR = Path('../../data-pipeline/data/processed/all_features')  # train.csv / val.csv / test.csv / feature_cols.json / scale_pos_weights.json

TRAIN_CSV = DATA_DIR / "train.csv"
VAL_CSV   = DATA_DIR / "val.csv"
TEST_CSV  = DATA_DIR / "test.csv"

FEATURE_COLS_JSON = DATA_DIR / "feature_cols.json"
SCALE_POS_WEIGHTS_JSON = DATA_DIR / "scale_pos_weights.json"

# ✅ (필수) 기존 scale_pos_weight 모델 pkl 경로
BASELINE_MODEL_PKL = '../models/xgboost/v2/24h/death.pkl'

# 출력 폴더
OUT_DIR = Path('../models/smote_vs_spw_outputs/death')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ✅ (필수) 타깃 컬럼 매핑 (본인 컬럼명으로 수정)
# 예시: {"DEATH": "death_next_24h", "VENT": "vent_start_next_24h", "PRESSOR": "pressor_start_next_24h", "COMPOSITE":"composite_next_24h"}
TARGETS = {
    "DEATH": "death_next_24h"
}

# Threshold sweep 설정 (네가 쓰던 값들)
THRESHOLDS = [0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50]
RANDOM_STATE = 42


In [ ]:

# ============================================
# Step 1: 데이터 로드 (✅ 너가 쓰던 로직 그대로)
# ============================================
print("\nStep 1: 데이터 로드")

# CSV 로드
train = pd.read_csv(TRAIN_CSV)
val   = pd.read_csv(VAL_CSV)
test  = pd.read_csv(TEST_CSV)

print(f"✓ Train: {len(train):,} rows")
print(f"✓ Val: {len(val):,} rows")
print(f"✓ Test: {len(test):,} rows")

# 피처 목록 로드
with open(FEATURE_COLS_JSON, "r") as f:
    feature_cols = json.load(f)
print(f"✓ 피처: {len(feature_cols)}개")

# 클래스 가중치 로드
with open(SCALE_POS_WEIGHTS_JSON, "r") as f:
    scale_pos_weights = json.load(f)
print(f"✓ 클래스 가중치 로드 완료")

# X 분리 (y는 train/val/test df에서 타깃 컬럼으로 접근)
X_train = train[feature_cols]
X_val   = val[feature_cols]
X_test  = test[feature_cols]

print(f"\n피처 shape:")
print(f"  X_train: {X_train.shape}")
print(f"  X_val:   {X_val.shape}")
print(f"  X_test:  {X_test.shape}")

# 레이블 분포 확인
print(f"\n=== 타겟 레이블 분포 (Train) ===")
for name, col in TARGETS.items():
    pos_rate  = train[col].mean() * 100
    pos_count = int(train[col].sum())
    weight    = scale_pos_weights.get(col, 1)
    print(f"  {name}: {pos_count:,} ({pos_rate:.2f}%) | scale_pos_weight={weight:.1f}")


In [ ]:

# ============================================
# Step 2: 유틸 (threshold sweep, 평가)
# ============================================
def threshold_sweep(y_true, y_prob, thresholds=THRESHOLDS):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)

    rows = []
    actual_pos = int(y_true.sum())

    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        pred_pos = int(y_pred.sum())

        tp = int(((y_pred == 1) & (y_true == 1)).sum())
        fp = int(((y_pred == 1) & (y_true == 0)).sum())
        fn = int(((y_pred == 0) & (y_true == 1)).sum())

        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall    = tp / (tp + fn) if (tp + fn) else 0.0
        f1        = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

        rows.append([t, precision, recall, f1, pred_pos, actual_pos])

    return pd.DataFrame(rows, columns=["Threshold","Precision","Recall","F1","PredPos","ActualPos"])


def predict_proba_xgb(model, X):
    # xgboost sklearn API
    return model.predict_proba(X)[:, 1]


def pretty_print_sweep(title, df):
    print("\n" + "="*70)
    print(title)
    print("="*70)
    print(df.to_string(index=False, formatters={
        "Threshold": lambda x: f"{x:>7.2f}",
        "Precision": lambda x: f"{x:>10.6f}",
        "Recall": lambda x: f"{x:>10.6f}",
        "F1": lambda x: f"{x:>10.6f}",
        "PredPos": lambda x: f"{int(x):,}",
        "ActualPos": lambda x: f"{int(x):,}",
    }))


In [ ]:

# ============================================
# Step 3: Baseline 모델 로드 (scale_pos_weight)
# ============================================
print("\nStep 3: Baseline 모델 로드")
with open(BASELINE_MODEL_PKL, "rb") as f:
    baseline_obj = pickle.load(f)

print("✓ Baseline object type:", type(baseline_obj))

# baseline_obj가 dict면 타깃별 모델을 가지고 있을 가능성이 높음
# - {target_col: model} 또는 {name: model} 둘 다 대응
def get_baseline_model_for_target(target_name, target_col):
    if isinstance(baseline_obj, dict):
        if target_col in baseline_obj:
            return baseline_obj[target_col]
        if target_name in baseline_obj:
            return baseline_obj[target_name]
        # 마지막 fallback: key들 출력
        raise KeyError(f"Baseline dict에서 {target_name=} / {target_col=} 키를 찾지 못함. keys={list(baseline_obj.keys())[:10]} ...")
    else:
        # 단일 모델이면 그대로 사용 (단, 이 경우 단일 타깃용이어야 함)
        return baseline_obj


In [ ]:

# ============================================
# Step 4: SMOTE 설정 & XGBoost 파라미터
# ============================================
# ✅ 추천: partial SMOTE (너무 과하게 1:1 만들지 않기)
SMOTE_SAMPLING_STRATEGY = 0.2  # minority:majority = 1:5 정도

smote = SMOTE(
    sampling_strategy=SMOTE_SAMPLING_STRATEGY,
    k_neighbors=5,
    random_state=RANDOM_STATE
)

# ✅ SMOTE 학습 모델은 scale_pos_weight 쓰지 않음(=1)
# 필요한 경우 여기서 하이퍼파라미터를 네 baseline과 맞춰도 됨
XGB_PARAMS_SMOTE = dict(
    n_estimators=600,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    min_child_weight=10,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    eval_metric="logloss",
)


In [ ]:

# ============================================
# Step 5: 타깃별로 성능 비교
#   - Baseline: pkl 로드 모델
#   - SMOTE: train에만 SMOTE 적용 후 재학습
# ============================================
all_rows = []

for name, target_col in TARGETS.items():
    print(f"\n\n########## Target: {name} ({target_col}) ##########")

    y_train = train[target_col].astype(int)
    y_val   = val[target_col].astype(int)
    y_test  = test[target_col].astype(int)

    # ---------- (A) Baseline 평가 ----------
    base_model = get_baseline_model_for_target(name, target_col)
    base_val_prob  = predict_proba_xgb(base_model, X_val)
    base_test_prob = predict_proba_xgb(base_model, X_test)

    base_ap_val  = average_precision_score(y_val, base_val_prob)
    base_ap_test = average_precision_score(y_test, base_test_prob)

    base_sweep_val  = threshold_sweep(y_val, base_val_prob)
    base_sweep_test = threshold_sweep(y_test, base_test_prob)

    pretty_print_sweep(f"[BASELINE][{name}] Threshold sweep (VAL) | AP={base_ap_val:.6f}", base_sweep_val)
    pretty_print_sweep(f"[BASELINE][{name}] Threshold sweep (TEST) | AP={base_ap_test:.6f}", base_sweep_test)

    # ---------- (B) SMOTE 학습 ----------
    # train에만 SMOTE 적용
    X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

    sm_model = XGBClassifier(**XGB_PARAMS_SMOTE)
    sm_model.fit(X_train_sm, y_train_sm)

    # 평가
    sm_val_prob  = predict_proba_xgb(sm_model, X_val)
    sm_test_prob = predict_proba_xgb(sm_model, X_test)

    sm_ap_val  = average_precision_score(y_val, sm_val_prob)
    sm_ap_test = average_precision_score(y_test, sm_test_prob)

    sm_sweep_val  = threshold_sweep(y_val, sm_val_prob)
    sm_sweep_test = threshold_sweep(y_test, sm_test_prob)

    pretty_print_sweep(f"[SMOTE][{name}] Threshold sweep (VAL) | AP={sm_ap_val:.6f}", sm_sweep_val)
    pretty_print_sweep(f"[SMOTE][{name}] Threshold sweep (TEST) | AP={sm_ap_test:.6f}", sm_sweep_test)

    # ---------- 결과 적재 ----------
    # VAL, TEST 각각 저장
    for split, ap, df_sweep, tag in [
        ("val", base_ap_val,  base_sweep_val,  "baseline"),
        ("test", base_ap_test, base_sweep_test, "baseline"),
        ("val", sm_ap_val,    sm_sweep_val,    "smote"),
        ("test", sm_ap_test,  sm_sweep_test,   "smote"),
    ]:
        tmp = df_sweep.copy()
        tmp.insert(0, "model", tag)
        tmp.insert(1, "target", name)
        tmp.insert(2, "split", split)
        tmp["AP"] = ap
        all_rows.append(tmp)

    # SMOTE 모델 저장(타깃별)
    sm_path = OUT_DIR / f"xgb_smote_{target_col}.pkl"
    with open(sm_path, "wb") as f:
        pickle.dump(sm_model, f)
    print(f"✓ Saved SMOTE model: {sm_path}")

# 비교표 저장
comparison_df = pd.concat(all_rows, ignore_index=True)
out_csv = OUT_DIR / "threshold_sweep_comparison.csv"
comparison_df.to_csv(out_csv, index=False)
print(f"\n✓ Saved comparison table: {out_csv}")
comparison_df.head()



## 참고
- Baseline pkl이 dict이면 `{target_col: model}` 또는 `{name: model}` 형태를 지원합니다.
- Baseline이 **단일 모델**이라면, `TARGETS`는 단일 타깃만 두는 것을 권장합니다.
- SMOTE는 **train set에만** 적용됩니다.


In [ ]:
# ============================================
# Step 6: "공정 비교" 추가
#  - 같은 PredPos 기준으로 baseline vs smote 비교
#  - (옵션) 같은 Recall 기준으로도 비교
# ============================================
import numpy as np
import pandas as pd

def metrics_at_threshold(y_true, y_prob, thr):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob)
    y_pred = (y_prob >= thr).astype(int)

    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall    = tp / (tp + fn) if (tp + fn) else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    predpos   = int(y_pred.sum())
    actualpos = int(y_true.sum())
    return {"thr": float(thr), "precision": precision, "recall": recall, "f1": f1,
            "predpos": predpos, "actualpos": actualpos}

def threshold_for_predpos(y_prob, target_predpos):
    y_prob = np.asarray(y_prob)
    target_predpos = int(target_predpos)
    target_predpos = max(1, min(target_predpos, len(y_prob)))

    # 상위 target_predpos개가 양성이 되게 하는 threshold
    order = np.argsort(-y_prob)
    thr = y_prob[order][target_predpos - 1]
    return float(thr)

def threshold_for_recall(y_true, y_prob, target_recall):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob)

    # 확률 내림차순으로 threshold를 내려가며 목표 recall 만족하는 최소 thr 찾기
    thresholds = np.unique(y_prob)[::-1]
    for thr in thresholds:
        y_pred = (y_prob >= thr).astype(int)
        tp = ((y_pred == 1) & (y_true == 1)).sum()
        fn = ((y_pred == 0) & (y_true == 1)).sum()
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        if recall >= target_recall:
            return float(thr)
    return float(thresholds[-1])  # 최저 threshold

# --- 타깃별 확률을 저장해두고 공정 비교 실행 ---
fair_rows = []

for name, target_col in TARGETS.items():
    y_val   = val[target_col].astype(int).values
    y_test  = test[target_col].astype(int).values

    # baseline 모델/확률
    base_model = get_baseline_model_for_target(name, target_col)
    base_val_prob  = predict_proba_xgb(base_model, X_val)
    base_test_prob = predict_proba_xgb(base_model, X_test)

    # smote 모델 로드(방금 저장한 파일) or 메모리에 sm_model이 있으면 그걸 써도 됨
    sm_path = OUT_DIR / f"xgb_smote_{target_col}.pkl"
    with open(sm_path, "rb") as f:
        sm_model = pickle.load(f)
    sm_val_prob  = predict_proba_xgb(sm_model, X_val)
    sm_test_prob = predict_proba_xgb(sm_model, X_test)

    # ---------- (A) PredPos 고정 비교 ----------
    # 기준 PredPos: baseline @0.50 의 predpos를 기준으로(원하면 바꿔도 됨)
    base_predpos_val = int((base_val_prob >= 0.50).sum())
    base_predpos_test = int((base_test_prob >= 0.50).sum())

    # VAL
    thr_base_val = 0.50
    thr_sm_val = threshold_for_predpos(sm_val_prob, base_predpos_val)
    fair_rows.append({
        "target": name, "split": "val", "mode": "fix_predpos",
        "baseline": metrics_at_threshold(y_val, base_val_prob, thr_base_val),
        "smote":    metrics_at_threshold(y_val, sm_val_prob,  thr_sm_val),
        "baseline_predpos_ref": base_predpos_val
    })

    # TEST
    thr_base_test = 0.50
    thr_sm_test = threshold_for_predpos(sm_test_prob, base_predpos_test)
    fair_rows.append({
        "target": name, "split": "test", "mode": "fix_predpos",
        "baseline": metrics_at_threshold(y_test, base_test_prob, thr_base_test),
        "smote":    metrics_at_threshold(y_test, sm_test_prob,  thr_sm_test),
        "baseline_predpos_ref": base_predpos_test
    })

    # ---------- (B) Recall 고정 비교 (옵션) ----------
    # 기준 Recall: baseline @0.50 의 recall
    base_recall_val = metrics_at_threshold(y_val, base_val_prob, 0.50)["recall"]
    base_recall_test = metrics_at_threshold(y_test, base_test_prob, 0.50)["recall"]

    # VAL
    thr_sm_val_r = threshold_for_recall(y_val, sm_val_prob, base_recall_val)
    fair_rows.append({
        "target": name, "split": "val", "mode": "fix_recall",
        "baseline": metrics_at_threshold(y_val, base_val_prob, 0.50),
        "smote":    metrics_at_threshold(y_val, sm_val_prob,  thr_sm_val_r),
        "baseline_recall_ref": base_recall_val
    })

    # TEST
    thr_sm_test_r = threshold_for_recall(y_test, sm_test_prob, base_recall_test)
    fair_rows.append({
        "target": name, "split": "test", "mode": "fix_recall",
        "baseline": metrics_at_threshold(y_test, base_test_prob, 0.50),
        "smote":    metrics_at_threshold(y_test, sm_test_prob,  thr_sm_test_r),
        "baseline_recall_ref": base_recall_test
    })

# 보기 좋게 펼치기
rows_flat = []
for r in fair_rows:
    base = r["baseline"]
    sm = r["smote"]
    rows_flat.append({
        "target": r["target"],
        "split": r["split"],
        "mode": r["mode"],
        "baseline_thr": base["thr"],
        "baseline_precision": base["precision"],
        "baseline_recall": base["recall"],
        "baseline_f1": base["f1"],
        "baseline_predpos": base["predpos"],
        "smote_thr": sm["thr"],
        "smote_precision": sm["precision"],
        "smote_recall": sm["recall"],
        "smote_f1": sm["f1"],
        "smote_predpos": sm["predpos"],
        **({ "ref_predpos": r.get("baseline_predpos_ref") } if "baseline_predpos_ref" in r else {}),
        **({ "ref_recall":  r.get("baseline_recall_ref") } if "baseline_recall_ref" in r else {}),
    })

fair_df = pd.DataFrame(rows_flat)
fair_out = OUT_DIR / "fair_comparison_fixed_predpos_or_recall.csv"
fair_df.to_csv(fair_out, index=False)
print(f"✓ Saved fair comparison: {fair_out}")
fair_df

